# 07 — End-to-end paper trading (dry run on synthetic data)

Wires every cognition + execution layer together and runs one
300-step episode. Reports per-stage latency and verifies the entire
system stays internally consistent.

In [1]:
from __future__ import annotations

import time
from dataclasses import dataclass

import numpy as np
import torch

from backend.cognition.agent.policy_network import PolicyNetwork
from backend.cognition.agent.ppo_agent import PPOAgent, PPOConfig
from backend.cognition.agent.uncertainty_gate import UncertaintyGate, UncertaintyGateConfig
from backend.config.constants import ACTION_DIM, NEXUS_OUTPUT_DIM
from backend.execution.broker.paper_adapter import PaperBroker
from backend.execution.safety.arbitrator import SafetyArbitrator
from backend.execution.safety.kill_switch import KillSwitchState
from backend.execution.safety.rules import SafetyContext
from backend.simulation.environments.base_env import EnvConfig, LuminaTradingEnv
from backend.simulation.generators.synthetic_data import SyntheticEpisodeGenerator

torch.manual_seed(0)
np.random.seed(0)

# The safety arbitrator emits a WARN log per veto. With an *untrained*
# random-policy agent we expect ~hundreds of vetoes per episode (the
# whole point — the arbitrator is keeping us safe). Quiet the logger
# for the duration of the notebook so the output remains readable.
import sys

from loguru import logger as _loguru

_loguru.remove()
_loguru.add(sys.stderr, level="ERROR")

DEVICE = "cpu"
EPISODE_LENGTH = 300

## 1. Build the environment

The environment owns the episode generator and produces (state,
reward) tuples. We use the GBM synthetic generator so prices are
realistic but cheap to sample.

In [2]:
generator = SyntheticEpisodeGenerator(
    n_steps=EPISODE_LENGTH,
    process="gbm",
    rng=np.random.default_rng(42),
)
env = LuminaTradingEnv(generator, EnvConfig(initial_capital=100_000.0))
state_dim = NEXUS_OUTPUT_DIM + env.config.portfolio_state_dim
print(f"Env observation dim: {state_dim}  (latent 256 + portfolio 4)")

Env observation dim: 260  (latent 256 + portfolio 4)


## 2. Build the cognitive agent

Untrained policy + uncertainty gate. We do NOT load a checkpoint —
this notebook tests the *plumbing*, not the *quality*.

In [3]:
policy = PolicyNetwork(state_dim=state_dim, action_dim=ACTION_DIM).to(DEVICE)
gate = UncertaintyGate(
    UncertaintyGateConfig(warmup_steps=10, rolling_window=10, threshold_high=0.85),
)
agent = PPOAgent(policy, gate, PPOConfig(), device=DEVICE)
print(f"Agent parameters: {sum(p.numel() for p in policy.parameters()):,}")

Agent parameters: 268,553


## 3. Build the execution stack

In the live system the execution orchestrator is invoked by the
end-to-end reflex loop. Here we drive it manually so we can record
per-stage latencies.

In [4]:
# A NO-OP "kill switch" backend — we don't need Redis for the dry-run.
class _NoopRedisCache:
    """Minimal stand-in for RedisCache used by KillSwitch."""

    class _Client:
        async def get(self, _key: str) -> bytes | None:
            return None

        async def set(self, _key: str, _value: str) -> bool:
            return True

    client = _Client()


@dataclass
class _DryRunKillSwitch:
    """Stand-in KillSwitch that is always in NORMAL state."""

    def get_state_sync(self) -> str:
        return KillSwitchState.NORMAL.value


broker = PaperBroker()
arbitrator = SafetyArbitrator()
kill_switch = _DryRunKillSwitch()
print(f"Initial equity: ${env.config.initial_capital:,.0f}")

Initial equity: $100,000


## 4. Episode loop

Per step we measure:
  * Agent latency   — `act()` including MC-Dropout
  * Safety latency  — arbitrator evaluation
  * Env latency     — environment step
A budget violation logs (no hard fail) so we can see distribution
tails clearly.

In [5]:
agent_latencies_ms: list[float] = []
safety_latencies_ms: list[float] = []
env_latencies_ms: list[float] = []
vetoes_total = 0

obs, _info = env.reset(seed=42)
equity_curve = [env.config.initial_capital]

for step_idx in range(EPISODE_LENGTH):
    # Stage 1 — agent forward (includes MC-Dropout, dominates latency)
    t0 = time.perf_counter()
    action, _log_prob, _value, uncertainty, _vetoed_by_gate = agent.act(obs)
    agent_latencies_ms.append((time.perf_counter() - t0) * 1000)

    # Stage 2 — safety arbitration
    t1 = time.perf_counter()
    ctx = SafetyContext(
        proposed_action=action,
        current_position=float(obs[NEXUS_OUTPUT_DIM]),  # first portfolio channel
        equity=env.config.initial_capital,  # paper broker, no live equity here
        peak_equity=env.config.initial_capital,
        uncertainty=float(uncertainty),
        kill_switch_state=kill_switch.get_state_sync(),
    )
    decision = arbitrator.evaluate(ctx)
    safety_latencies_ms.append((time.perf_counter() - t1) * 1000)
    if not decision.approved:
        vetoes_total += 1
        # Replace the action with a flat-out (the canonical defensive action).
        action = np.zeros_like(action)

    # Stage 3 — env step
    t2 = time.perf_counter()
    obs, _reward, terminated, truncated, info = env.step(action)
    env_latencies_ms.append((time.perf_counter() - t2) * 1000)
    equity_curve.append(info.get("equity", equity_curve[-1]))

    if terminated or truncated:
        print(f"Episode ended early at step {step_idx} (terminated={terminated})")
        break

Episode ended early at step 298 (terminated=False)


## 5. Latency summary

In [6]:
def _pctiles(values: list[float]) -> tuple[float, float, float]:
    arr = np.asarray(values)
    return (
        float(np.percentile(arr, 50)),
        float(np.percentile(arr, 95)),
        float(np.percentile(arr, 99)),
    )


stages = {
    "agent.act      ": agent_latencies_ms,
    "safety.evaluate": safety_latencies_ms,
    "env.step       ": env_latencies_ms,
}
print(f"\nLatency over {len(agent_latencies_ms)} steps (ms):")
print(f"  {'stage':16s}  {'p50':>7s}  {'p95':>7s}  {'p99':>7s}")
for stage, vals in stages.items():
    p50, p95, p99 = _pctiles(vals)
    print(f"  {stage}  {p50:7.3f}  {p95:7.3f}  {p99:7.3f}")

total_per_step = [
    a + s + e
    for a, s, e in zip(agent_latencies_ms, safety_latencies_ms, env_latencies_ms, strict=True)
]
t_p99 = float(np.percentile(total_per_step, 99))
print(
    f"  {'TOTAL          ':16s}  {np.percentile(total_per_step, 50):7.3f}  "
    f"{np.percentile(total_per_step, 95):7.3f}  {t_p99:7.3f}"
)


Latency over 299 steps (ms):
  stage                 p50      p95      p99
  agent.act          2.273    3.243    3.769
  safety.evaluate    0.018    0.026    0.035
  env.step           0.024    0.034    0.036
  TOTAL               2.313    3.297    3.814


## 6. Final state + acceptance assertions

In [7]:
final_equity = float(equity_curve[-1])
print(f"\nFinal equity:    ${final_equity:,.2f}")
print(f"Vetoes by arbitrator: {vetoes_total}")
print(f"p99 total step latency: {t_p99:.2f} ms")

# Wide PnL band — the agent is untrained random noise.
assert 50_000.0 <= final_equity <= 200_000.0, (
    f"Final equity {final_equity:.2f} outside the sanity band [50k, 200k]; "
    "likely indicates an arithmetic bug rather than bad strategy."
)
assert vetoes_total >= 0
assert t_p99 < 50.0, f"p99 total step latency {t_p99:.2f} ms exceeds the 50 ms compute budget"
print("\nPASS — full stack composes and stays within budgets.")


Final equity:    $101,900.47
Vetoes by arbitrator: 237
p99 total step latency: 3.81 ms

PASS — full stack composes and stays within budgets.
